<a href="https://colab.research.google.com/github/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/LLM_Agentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agentes para LLMS**

Agentes (Agents) em LLMs são estruturas que permitem que um modelo de linguagem
observe o estado atual, decida uma ação, execute ferramentas externas e continue
interagindo de forma iterativa. Diferentemente de uma simples chamada a um LLM,
agentes são capazes de:


- Raciocinar de forma iterativa (loop perception → reasoning → act → observe).
- Acessar ferramentas (APIs, bancos de dados, calculadoras, buscadores, arquivos).
- Utilizar memória para contexto longo.
- Selecionar ações autonomamente.

# LangChain, LangGraph, LangSmith e LangFlow  

* **LangChain (2022)**: A base para criar fluxos de trabalho LLM modulares (instruções, ferramentas, memória, recuperadores) usando APIs simples como LCEL. Prototipagem e aplicativos lineares.

>> **LangGraph (2024)**: **O orquestrador, gerencia fluxos de trabalho complexos, com estado e ramificados, com repetições, loops e persistência.**

* **LangSmith**: Ferramenta independente de framework para rastreamento, avaliação e monitoramento de aplicativos LLM. Para depuração, testes de regressão e controle de qualidade.

* **LangFlow**: O construtor visual e exportação para código.

O LangGraph é a evolução do LangChain para **construção de agentes, fluxos
estruturados, multi-step reasoning e orquestração**. É inspirado em sistemas
reativos, máquinas de estado e workflows determinísticos.

![image](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/LangGraph0.avif?raw=true)


Padrões comuns em LangGraph:
- **Nodes**: funções que processam inputs e produzem outputs.

> - Invocando um modelo de linguagem amplo (LLM)
> - Chamar uma ferramenta ou API
> - Executando uma função Python personalizada
> - Lógica de roteamento ou decisões de ramificação

- **Edges**: regras de transição entre nós. **If's e Loops**.

- **State**: dicionário ou modelo Pydantic que mantém o estado.

> - Adiante, vários tipos de armazenamento.

- **Tool calling**: suportado nativamente em modelos OpenAI/HF.

> * Uma ferramenta é qualquer função externa ou interna que um agente pode chamar, como uma **pesquisa na web**, uma calculadora ou planilha eletrônica, um utilitário personalizado, uma ferramenta e **e-mail**.

- **Loops de raciocínio**: implementados com grafos cíclicos.

<br>

![image](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/LangGraph1.avif?raw=true)

<br>

![image](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/LangGraph2.avif?raw=true)

<br>

Existem também diversos tipos de mensagens, HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMassage sendo um conceito útilo na hora de implementar um agente com o LangGraph.

# Exemplo simples

In [25]:
!pip install langgraph langchain_community langchain-core langchain_openai


In [29]:
from langchain_openai import ChatOpenAI
import os

# Adicione sua chave de API da OpenAI aqui
os.environ["OPENAI_API_KEY"] = "API KEY"
# no final substitua pela 1a palavra do alfabeto em maiúsculo

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### `StateGraph` $⟹$ `add_node` $⟹$ `add_edge` $⟹$ `END` $⟹$ `compile`

In [31]:
# load_dotenv() # Obtaining over secret keys, ou use aciam

# Creation of the state using a Typed Dictionary
class AgentState(TypedDict):
  messages: List[HumanMessage] # We are going to be storing Human Messages (the user input) as a list of messages

llm = ChatOpenAI(model="gpt-4o") # Our model choice

# This is an action - the underlying function of our node
def process(state: AgentState) -> AgentState:
  response = llm.invoke(state["messages"])
  print(f"\nAI: {response.content}")
  return state

graph = StateGraph(AgentState) # Initialization of a Graph

graph.add_node("process_node", process) # Adding nodes

graph.add_edge(START, "process_node") # Adding edges
graph.add_edge("process_node", END)

agent = graph.compile() # Compiling the graph

### $⟹$ `Agent.invoke()`

In [ ]:
user_input = input("Enter: ")
while user_input != "exit":
  agent.invoke({"messages": [HumanMessage(content=user_input)]})
  user_input = input("Enter: ")

### Armazenamento

- SQLite
- PostgreSQL
- Amazon S3, Blobs do Azure, nuvem do Google
- **Memória** (dados de sessão)

Podem haver estados de curto (dados de sessão, em geral em memória) ou longo prazo. Também são fornecidos métodos de limpeza (`RemoveMessage()`) e redução dos dados como apara (`trim_messages()`), sumarização (`SummarizationNode()`)


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
memory = SqliteSaver.from_conn_string(":memory:") # This is for connecting to the SQLite database
graph = graph_builder.compile(checkpointer=memory) # Compiling graph with checkpointer as SQLite backend

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph

checkpointer = InMemorySaver()

builder = StateGraph(...)
graph = builder.compile(checkpointer=checkpointer)

# Invoking the Graph with our message
agent_graph.invoke(
    {"messages": [{"role": "user", "content": "What's the weather today?"}]},
    {"configurable": {"thread_id": "session_42"}},
)

# ReAct Agent

Um agente ReAct é um agente de IA que utiliza a estrutura de "raciocínio e ação" (ReAct) para combinar o raciocínio em cadeia (CoT) com o uso de ferramentas externas. A estrutura ReAct aprimora a capacidade de um modelo de linguagem de grande porte (LLM) de lidar com tarefas complexas e tomada de decisões em fluxos de trabalho com agentes. É a ideia por traz de ferramentas atuais como o **LangGraph, AutoGPT, CrewAI, OpenAI Assistants API**.

<br>

![image](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/ReAct.png?raw=true)

<br>

Apresentado inicialmente por Yao e outros no artigo de 2023, **ReACT: Sinergizando Raciocínio e Ação em Modelos de Linguagem**, o ReAct pode ser entendido, de forma geral, como um paradigma de aprendizado de máquina  (ML) para integrar as capacidades de raciocínio e tomada de ação dos Modelos de Linguagem (LLMs).

Antes do ReAct, os modelos funcionavam assim:

> **Usuário → LLM → Resposta final**

Limitações:

- Sem acesso a ferramentas.

- Sem 'memória de passos'.

- Não consegume corrigir erros durante o raciocínio.

Com o ReAct:

> **Usuário → LLM (Reasoning) → Ferramenta (Action) → Observação → Novo Reasoning → ... → Resposta final**

## React incentivando
O ReAct prompting é uma técnica específica de prompting projetada para guiar um LLM (Learning Learning Machine - Máquina de Aprendizagem Baseada em Aprendizagem) a seguir o paradigma ReAct de ciclos de pensamento , ação e observação . Embora o uso explícito de métodos convencionais de prompting ReAct não seja estritamente necessário para construir um agente ReAct, a maioria dos agentes baseados em ReAct implementa ou, pelo menos, se inspira diretamente nele.

Descrita inicialmente no artigo original do ReAct, a função principal do recurso de prompts do ReAct é instruir um LLM (Learning Learning Machine) a seguir o loop do ReAct e estabelecer quais ferramentas podem ser usadas — ou seja, quais ações podem ser tomadas — ao lidar com as consultas do usuário.

### Exemplos de `prompts`

In [ ]:
# qwery writer prompt - model feeding prompt
query_writer_prompt="""Your goal is to generate targeted web search query.
The query will gather information related to a specific topic.

Topic:
{research_topic}

Return your query as a JSON object:
{{
    "query": "string",
    "aspect": "string",
    "rationale": "string"
}}
"""
# rationale - why this query is important. incourages model to think about generation of a query itself
query_writer_prompt

In [ ]:
# summarizer instructions prompt - 1st summarisation
summarizer_instructions_prompt="""Your goal is to generate a high-quality summary of the web search results.

When EXTENDING an existing summary:
1. Seamlessly integrate new information without repeating what's already covered
2. Maintain consistency with the existing content's style and depth
3. Only add new, non-redundant information
4. Ensure smooth transitions between existing and new content

When creating a NEW summary:
1. Highlight the most relevant information from each source
2. Provide a concise overview of the key points related to the report topic
3. Emphasize significant findings or insights
4. Ensure a coherent flow of information

In both cases:
- Focus on factual, objective information
- Maintain a consistent technical depth
- Avoid redundancy and repetition
- DO NOT use phrases like "based on the new results" or "according to additional sources"
- DO NOT add a preamble like "Here is an extended summary ..." Just directly output the summary.
- DO NOT add a References or Works Cited section.
"""

summarizer_instructions_prompt

In [ ]:
# reflection prompt - for agents internal thinking
reflection_instructions_prompt = """You are an expert research assistant analyzing a summary about {research_topic}.

Your tasks:
1. Identify knowledge gaps or areas that need deeper exploration
2. Generate a follow-up question that would help expand your understanding
3. Focus on technical details, implementation specifics, or emerging trends that weren't fully covered

Ensure the follow-up question is self-contained and includes necessary context for web search.

Return your analysis as a JSON object:
{{
    "knowledge_gap": "string",
    "follow_up_query": "string"
}}"""

reflection_instructions_prompt

## Prompt ReAct mínimo (manual)


```
prompt = '''
Você é um agente que alterna entre pensamento e ação para resolver problemas.
Siga SEMPRE o formato abaixo:

Thought: descreva seu raciocínio sobre o que fazer
Action: escolha UMA ação entre {search, calculator, lookup} com argumentos
Observation: eu fornecerei o resultado da ação
... (repita Thought/Action/Observation quantas vezes quiser)
Final Answer: forneça a resposta final ao usuário

Importante:
- Thought é seu raciocínio interno.
- NÃO pule diretamente para a resposta final.
- O ciclo Thought → Action → Observation deve acontecer antes da resposta final.
'''


**Usuário**

- **Pergunta**: "Qual é o país de nascimento de Albert Einstein?"

**LLM**

- **Thought**: preciso descobrir onde Einstein nasceu; usar search.

- **Action**: search("Albert Einstein birthplace")

>> Observation: Albert Einstein nasceu em Ulm, Alemanha.

- **Thought**: já tenho a informação.
- **Final Answer**: Albert Einstein nasceu em Ulm, na Alemanha.

O exemplo clássico do paper original (HotpotQA), é algo semelhante, que combina diferentes pesquisas (duas) para responder à pergunta clássica do paper:

***What is the elevation range for the area that the eastern part of Mount Timpanogos drains into?***